# Order flow and intraday profile

Thin wrapper over `ratesresearch.flow`: the committed preset `usd_rfq_flow` was fitted from nine business days of DTCC SDR public dissemination (USD SOFR OIS, 2025-10-01 to 2025-10-14 without 2025-10-13), whose manifest is `data/manifests/dtcc_rates_2025-10.json`. Raw files are not in the repository (DR-006); this notebook reads the fitted preset and the FINRA-layout synthetic fixture.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from ratesresearch import backtest_analysis as ba
from ratesresearch import presets

ROOT = Path.cwd()
while not (ROOT / "data" / "presets.yaml").is_file():
    ROOT = ROOT.parent
PRESETS = ROOT / "data" / "presets.yaml"


def show(df: pd.DataFrame) -> None:
    display(Markdown(ba.to_markdown(df)))

In [ ]:
from ratesresearch import flow

p = presets.get_preset(PRESETS, "usd_rfq_flow")
show(pd.DataFrame(flow.profile_table(p), columns=["time (New York)", "trades per minute"]).round(3))
show(
    pd.DataFrame(
        [{k: v["value"] for k, v in p["parameters"].items() if k.startswith(("size", "tenor"))}]
    ).round(4)
)

In [ ]:
from ratesresearch.data import finra

fx = finra.load_treasury_trades(ROOT / "data/fixtures/finra/treasury_trades_fixture.csv")
fit = flow.fit_finra_fixture(fx, "fixture")
display(Markdown("FINRA status: " + finra.STATUS))
show(pd.DataFrame([{k: v["value"] for k, v in fit["parameters"].items()}]).round(4))